# Make PDB File for DMS Data Visualization

OG Script written by Matthew Chan, adapted by RBakker 13 Mar 2025

- Loads PDB file and replaces B factor column with LN DMS Data for Visualization for Figure 7

### Load Libraries

In [1]:
from Bio.PDB import *
import numpy as np
import pandas as pd


### Define Variables

In [2]:
pdb_file = "1qfq.pdb"
scores = "../tables/dms_data.csv"
out_file = "../data/pdb/output.pdb"

###  Function to Reset B Factor Column

In [3]:
def set_b_factor_to_zero(structure, min_value = -10):
	# Set b factor of entire protein to large negative number if using 3-point scale.
	# Alternative can be set to 0
	for model in structure:
		for chain in model:
			for residue in chain:
				for atom in residue:
					atom.set_bfactor(min_value)


### Read in PDB File, Change B Factor Column and Output

In [5]:
parser = PDBParser()
structure = parser.get_structure(" ", pdb_file)

# First set all b_factor values to 0 or negative number
set_b_factor_to_zero(structure)


# Read DMS csv into dataframe
df = pd.read_csv(scores)
for index, row in df.iterrows():
	resid = row["codon_position"]
	lfc = row["mean_lfc"]

	# Iterate through each atom of each DMS position 
	# and change the b factor
	for model in structure:
		chain = model["B"]					# change chain ID to protein of interest	
		for atom in chain[int(resid)]: 		# loop through all atoms in the residue, chain
			atom.set_bfactor(lfc)			# change the b factor for all the atoms in the residue


io=PDBIO()
io.set_structure(structure)
io.save(out_file)